---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---


# Notebook 00 — LAS File Metadata Extraction

**Objective:** Extract header information from all LAS files for wells in the
Sergipe-Alagoas Basin and consolidate into a reference CSV file.

**Output:** `data/raw/well_coordinates.csv`

**Extracted fields:** well identification, geographic and UTM coordinates,
depth references (KB, GL, DF), water depth (WDEP), logged interval
(STRT, STOP), operator, and field.

**Derived fields:** logged interval, sedimentary column above the top of
the logged section — used as a burial proxy in the geological analysis (Note 14.1).

## 0. Imports and Configuration

In [ ]:
import os
import re
import lasio
import numpy as np
import pandas as pd
from pathlib import Path
from pyproj import Transformer
from scipy.spatial import cKDTree

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
LAS_DIR    = Path('E:/GitHub/sonic_log_prediction/data/raw/LAS')
OUTPUT_DIR = Path('E:/GitHub/sonic_log_prediction/data/raw')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / 'well_coordinates.csv'

print(f'LAS folder : {LAS_DIR}')
print(f'Output     : {OUTPUT_CSV}')

## 1. Loading LAS Files

In [ ]:
# ── Locate all LAS files recursively ─────────────────────────────────────────
las_files = sorted(LAS_DIR.rglob('*.las')) + sorted(LAS_DIR.rglob('*.LAS'))
las_files = sorted(set(las_files))  # deduplicate

print(f'LAS files found: {len(las_files)}')
for f in las_files:
    print(f'  {f}')

In [ ]:
# ── Load LAS files ────────────────────────────────────────────────────────────
las_loaded = []
errors     = []

for path in las_files:
    try:
        las = lasio.read(str(path), ignore_header_errors=True)
        las_loaded.append((path.name, path, las))
    except Exception as e:
        errors.append((path.name, str(e)))

print(f'Successfully loaded : {len(las_loaded)}')
print(f'Read errors         : {len(errors)}')
if errors:
    for name, err in errors:
        print(f'{name}: {err[:80]}')

## 2. Header Extraction

In [ ]:
# ── Inspect available fields in the first file ────────────────────────────────
if las_loaded:
    _, _, las_sample = las_loaded[0]
    print('Available fields in header (first file):')
    print(f'  {"Mnemonic":<12} {"Unit":<10} {"Value":<30} Description')
    print('  ' + '-' * 75)
    for item in las_sample.well:
        print(f'  {item.mnemonic:<12} {str(item.unit):<10} '
              f'{str(item.value):<30} {item.descr}')

In [ ]:
# ── Helper functions: safe numeric and string conversion ──────────────────────
def safe_num(value, null_vals=(-999.25, -9999.25, 9999.25)):
    """Converts to float; returns NaN if null or sentinel value."""
    try:
        v = float(str(value).strip().replace(',', '.'))
        return np.nan if v in null_vals or abs(v) > 1e6 else v
    except (ValueError, TypeError):
        return np.nan

def safe_str(value):
    """Converts to a clean string; returns None if empty."""
    s = str(value).strip() if value is not None else ''
    return s if s and s.lower() not in ('none', 'null', '') else None

def get_field(las, *mnemonics):
    """Returns the value of the first mnemonic found in the header."""
    well_dict = {item.mnemonic.strip().upper(): item for item in las.well}
    for m in mnemonics:
        if m.upper() in well_dict:
            return well_dict[m.upper()].value
    return None

In [ ]:
# ── Extract all fields of interest ───────────────────────────────────────────
records = []

for filename, path, las in las_loaded:

    # Identification
    well = safe_str(get_field(las, 'WELL', 'WN'))
    uwi  = safe_str(get_field(las, 'UWI', 'API', 'LIC'))
    comp = safe_str(get_field(las, 'COMP', 'OPERATOR', 'OPER'))
    fld  = safe_str(get_field(las, 'FLD', 'FIELD'))
    loc  = safe_str(get_field(las, 'LOC', 'LOCATION'))
    ctry = safe_str(get_field(las, 'CTRY', 'COUNTRY', 'COUNT'))
    prov = safe_str(get_field(las, 'PROV', 'STATE'))
    cnty = safe_str(get_field(las, 'CNTY', 'COUNTY'))
    date = safe_str(get_field(las, 'DATE', 'SRVC', 'SDAT'))

    # Geographic coordinates
    lati = safe_str(get_field(las, 'LATI', 'LAT', 'LATITUDE'))
    long = safe_str(get_field(las, 'LONG', 'LON', 'LONGITUDE'))

    # Depth references
    strt = safe_num(get_field(las, 'STRT', 'START'))
    stop = safe_num(get_field(las, 'STOP', 'END'))
    step = safe_num(get_field(las, 'STEP'))
    kb   = safe_num(get_field(las, 'KB', 'EREF', 'EPDB'))
    gl   = safe_num(get_field(las, 'GL', 'GREF', 'GRND'))
    df   = safe_num(get_field(las, 'DF', 'DATUM'))
    wdep = safe_num(get_field(las, 'WDEP', 'WDTH', 'WDEP', 'WD'))
    egl  = safe_num(get_field(las, 'EGL', 'ELEV'))
    tdd  = safe_num(get_field(las, 'TDD', 'TD'))

    # Null value
    null = safe_num(get_field(las, 'NULL', 'NVAL'))

    records.append({
        'File'       : filename,
        'Well_Name'  : well,
        'UWI'        : uwi,
        'Company'    : comp,
        'Field'      : fld,
        'Location'   : loc,
        'Country'    : ctry,
        'Province'   : prov,
        'County'     : cnty,
        'Date'       : date,
        'Latitude'   : lati,
        'Longitude'  : long,
        'STRT'       : strt,
        'STOP'       : stop,
        'STEP'       : step,
        'KB'         : kb,
        'GL'         : gl,
        'DF'         : df,
        'WDEP'       : wdep,
        'EGL'        : egl,
        'TDD'        : tdd,
        'NULL_VAL'   : null,
    })

df_meta = pd.DataFrame(records)
print(f'Records extracted: {len(df_meta)}')
print(f'Columns: {list(df_meta.columns)}')
df_meta.head()

## 3. Cleaning and Derived Fields

In [ ]:
# ── KB cleanup ────────────────────────────────────────────────────────────────
def resolve_kb(row):
    kb = row['KB']
    df = row['DF']
    if pd.notna(kb) and kb >= 10 and kb != -999.25:
        return kb
    if pd.notna(df) and df >= 10:
        return df
    return 25.0

df_meta['KB'] = df_meta.apply(resolve_kb, axis=1)

# ── Convert coordinates to decimal degrees ────────────────────────────────────
def parse_coordinate(val):
    """
    Converts a coordinate from multiple formats to decimal degrees.
    Accepts: decimal string, DMS (e.g., -10 30 25.5), DDMMSS.ss
    """
    if val is None:
        return np.nan
    s = str(val).strip().replace(',', '.')
    # Already decimal
    try:
        return float(s)
    except ValueError:
        pass
    # Space-separated DMS format
    parts = re.split(r'[\s°\'"]+', s.replace('-', ''))
    sign  = -1 if s.strip().startswith('-') else 1
    try:
        d = float(parts[0])
        m = float(parts[1]) if len(parts) > 1 else 0
        sec = float(parts[2]) if len(parts) > 2 else 0
        return sign * (d + m / 60 + sec / 3600)
    except (ValueError, IndexError):
        return np.nan

df_meta['Lat_dd']  = df_meta['Latitude'].apply(parse_coordinate)
df_meta['Lon_dd']  = df_meta['Longitude'].apply(parse_coordinate)

# ── Convert to UTM (zone 24S — Sergipe-Alagoas Basin) ────────────────────────
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:31984',
                                    always_xy=True)

def to_utm(row):
    try:
        x, y = transformer.transform(row['Lon_dd'], row['Lat_dd'])
        return pd.Series({'UTM_E': round(x, 2), 'UTM_N': round(y, 2)})
    except Exception:
        return pd.Series({'UTM_E': np.nan, 'UTM_N': np.nan})

utm = df_meta.apply(to_utm, axis=1)
df_meta['UTM_E'] = utm['UTM_E']
df_meta['UTM_N'] = utm['UTM_N']

print('Coordinates converted.')
print(df_meta[['Well_Name','Lat_dd','Lon_dd','UTM_E','UTM_N']])

In [ ]:
# ── Derived depth fields ──────────────────────────────────────────────────────

# Logged interval
df_meta['logged_interval_m'] = df_meta['STOP'] - df_meta['STRT']

# Water depth: use WDEP if available; fallback: negative GL
df_meta['water_depth_m'] = df_meta.apply(
    lambda r: r['WDEP'] if pd.notna(r['WDEP']) and r['WDEP'] > 0
              else (abs(r['GL']) if pd.notna(r['GL']) and r['GL'] < 0
              else np.nan),
    axis=1
)

# Sedimentary column above the top of the logged section
# = top of log (STRT) - KB + water depth
# Represents the total column (water + sediment) above the first
# logged depth — used as a burial proxy in the geological analysis
df_meta['overburden_thickness_m'] = (
    df_meta['STRT'] - df_meta['KB'] + df_meta['water_depth_m']
)

print('Derived fields calculated.')
cols_check = ['Well_Name', 'STRT', 'STOP', 'KB', 'GL', 'WDEP',
              'water_depth_m', 'overburden_thickness_m', 'logged_interval_m']
print(df_meta[cols_check].sort_values('STRT').to_string(index=False))

In [ ]:
# ── Investigate wells without GL ──────────────────────────────────────────────
sem_gl = df_meta[df_meta['GL'].isna()][
    ['Well_Name', 'STRT', 'STOP', 'KB', 'DF', 'GL', 'WDEP',
     'Lat_dd', 'Lon_dd', 'UTM_E', 'UTM_N']]
print('Wells without GL:')
print(sem_gl.to_string(index=False))

print()

# Nearest neighboring wells for each case (by UTM Euclidean distance)
from scipy.spatial import cKDTree

df_com_gl  = df_meta[df_meta['water_depth_m'].notna()].copy()
df_sem_gl  = df_meta[df_meta['water_depth_m'].isna()].copy()

if len(df_com_gl) > 0 and len(df_sem_gl) > 0:
    tree = cKDTree(df_com_gl[['UTM_E', 'UTM_N']].values)
    dists, idxs = tree.query(df_sem_gl[['UTM_E', 'UTM_N']].values, k=3)

    print('Nearest neighbors (to estimate water_depth):')
    for i, row in df_sem_gl.iterrows():
        print(f'\n  {row["Well_Name"]}:')
        for j, (dist, idx) in enumerate(zip(dists[list(df_sem_gl.index).index(i)],
                                             idxs[list(df_sem_gl.index).index(i)])):
            viz = df_com_gl.iloc[idx]
            print(f'    {j+1}. {viz["Well_Name"]:25s} '
                  f'dist={dist/1000:.1f}km  '
                  f'water_depth={viz["water_depth_m"]:.0f}m')

In [ ]:
# ── Interpolate water_depth by geographic neighborhood ────────────────────────
df_com_gl = df_meta[df_meta['water_depth_m'].notna()].copy()
df_sem_gl = df_meta[df_meta['water_depth_m'].isna()].copy()

tree = cKDTree(df_com_gl[['UTM_E', 'UTM_N']].values)
dists, idxs = tree.query(df_sem_gl[['UTM_E', 'UTM_N']].values, k=3)

for i, (orig_idx, row) in enumerate(df_sem_gl.iterrows()):
    d = dists[i]
    # Inverse distance weighted average (IDW)
    weights = 1 / (d + 1e-6)
    weights = weights / weights.sum()
    wdep_interp = sum(
        weights[j] * df_com_gl.iloc[idxs[i][j]]['water_depth_m']
        for j in range(3)
    )
    df_meta.loc[orig_idx, 'water_depth_m'] = round(wdep_interp, 1)
    print(f'{row["Well_Name"]:20s} interpolated water_depth = '
          f'{wdep_interp:.1f}m '
          f'(neighbors: {dists[i][0]/1000:.1f}, '
          f'{dists[i][1]/1000:.1f}, '
          f'{dists[i][2]/1000:.1f} km)')

# ── Recalculate overburden_thickness_m with interpolated values ──────────────────────
df_meta['overburden_thickness_m'] = (
    df_meta['STRT'] - df_meta['KB'] + df_meta['water_depth_m']
)

print()
print('All wells sorted by overburden_thickness_m:')
print(df_meta[cols + ['R2']].sort_values('overburden_thickness_m')
                             .to_string(index=False)
      if 'R2' in df_meta.columns else
      df_meta[cols].sort_values('overburden_thickness_m').to_string(index=False))

## 6. Save

In [ ]:
# ── Save final CSV ────────────────────────────────────────────────────────────
df_meta.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'Saved: {OUTPUT_CSV}')
print(f'Shape: {df_meta.shape}')
print(f'Columns: {list(df_meta.columns)}')
print()

# Check fields with missing values
missing = df_meta.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print('Fields with missing values:')
    for col, n in missing.items():
        print(f'  {col:<25}: {n:2d} / {len(df_meta)} '
              f'({n/len(df_meta)*100:.0f}%)')